In [1]:
import os
from dotenv import load_dotenv

print("Exists:", os.path.exists("../openai.env"))

load_dotenv("../openai.env", override=True)
print("Key loaded:", bool(os.getenv("OPENAI_API_KEY")))

Exists: True
Key loaded: True


In [59]:
from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv

In [60]:
load_dotenv("../openai.env")

True

# 1. Loading Documents


- This is the python download script for wikipedia
```python

import requests

topics = ["Photosynthesis", "Water cycle", "Cellular respiration",
          "Plate tectonics", "Food chain"]

out_dir = "../resources"          # notebooks/ -> SemProject_MLEngine/resources
os.makedirs(out_dir, exist_ok=True)

headers = {"User-Agent": "SemProject-MLEngine/1.0 (student project)"}

for title in topics:
    r = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={"action": "query", "prop": "extracts", "explaintext": 1,
                "titles": title, "redirects": 1, "format": "json"},
        headers=headers, timeout=30,
    )
    page = next(iter(r.json()["query"]["pages"].values()))
    words = page["extract"].split(" ")
    text = " ".join(words[:1500])

    fname = f"{out_dir}/{title.lower().replace(' ', '_')}.txt"
    with open(fname, "w", encoding="utf-8") as f:
        f.write(f"Source: Wikipedia, '{title}' (CC BY-SA)\n\n{text}")
    print(f"Saved {fname} ({len(text.split())} words)")

```


In [61]:
docs_path ="../../RAG_CONTEXT_DOCUMENTS"
loader= DirectoryLoader(docs_path, glob="*.pdf", loader_cls=PyPDFLoader, show_progress=True,)

documents=loader.load()
print(f"Loaded {len(documents)} documents")

 12%|█▎        | 1/8 [00:04<00:29,  4.21s/it]fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/AEYEUS+MyriadPro-BoldCond', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 86, '/FontDescriptor': IndirectObject(1574, 0, 137357452040464), '/LastChar': 117, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [452, 0, 0, 402, 0, 0, 0, 0, 0, 0, 0, 0, 0, 323, 0, 419, 0, 0, 0, 223, 0, 0, 0, 0, 0, 431, 0, 0, 294, 0, 0, 452]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/AEYEUS+MyriadPro-BoldCond', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 86, '/FontDescriptor': IndirectObject(1574, 0, 137357452040464), '/LastChar': 117, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [452, 0, 0, 402, 0, 0, 0, 0, 0, 0, 0, 0, 0, 323, 0, 419, 0, 0, 0, 223, 0, 0, 0, 0, 0, 431, 0, 0, 294, 0, 0, 452]}, but is no

Loaded 758 documents


- There are 758 total pages loaded, each page in pdf registers as a document

## 2. Splitting Documents into Chunks

### a) Fixed Chunking

In [63]:
fixed_splitter= CharacterTextSplitter(separator="",chunk_size=1000, chunk_overlap=100)
fixed_chunks= fixed_splitter.split_documents(documents)

chunk_lengths= []
for chunk in fixed_chunks:
    chunklength=len(chunk.page_content)
    chunk_lengths.append(chunklength)
print(chunk_lengths[:50])
print (f"Created {len(chunk_lengths)} chunks")

[209, 953, 1000, 1000, 1000, 1000, 1000, 482, 1000, 999, 702, 1000, 1000, 819, 999, 1000, 1000, 571, 1000, 1000, 1000, 463, 1000, 999, 730, 1000, 998, 948, 999, 1000, 999, 585, 1000, 998, 999, 267, 1000, 1000, 1000, 411, 1000, 999, 1000, 628, 1000, 794, 143, 39, 78, 513]
Created 1652 chunks


In [70]:
for i, chunk in enumerate(fixed_chunks[:10]):
    print(f"----Chunk {i+1}----")
    print(chunk.page_content)
    print()

----Chunk 1----
LAWS OF KENYA
THE ELECTION OFFENCES ACT
CAP. 66
Revised and published by the National Council for Law Reporting
with the authority of the Attorney-General as gazetted by the Government Printer
www.kenyalaw.org

----Chunk 2----
Kenya
Election Offences Act
Cap. 66
Legislation as at 31 December 2022
 
 
 
 
 
 
 
 
 
 
 
 
By Kenya Law and Laws.Africa. Share widely and freely.
www.kenyalaw.org | info@kenyalaw.org
FRBR URI: /akn/ke/act/2016/37/eng@2022-12-31
There is no copyright on the legislative content of this document.
This PDF copy is licensed under a Creative Commons Attribution NonCommercial ShareAlike 4.0 License (CC BY-NC-
SA 4.0). This license enables reusers to distribute, remix, adapt, and build upon the material in any medium or format
for noncommercial purposes only, and only so long as attribution is given to the creator. If you remix, adapt, or build
upon the material, you must license the modiﬁed material under identical terms. CC BY-NC-SA includes the fol

- The general observation is useless parts get chunked, such as the table of contents

### b) Recursive Splitting

In [71]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
recursive_splitter=RecursiveCharacterTextSplitter(separators=["\n\n","\n"," ",""], chunk_size=500, chunk_overlap=50)

In [72]:
recursive_chunks=recursive_splitter.split_documents(documents)

In [75]:
chunksizes=[]
for recursive_chunk in recursive_chunks:
    chunksizes.append(len(recursive_chunk.page_content))

print(chunksizes[:10])
print(len(chunksizes))
for i, recursive_chunk in enumerate(recursive_chunks[:10]):
    print(f"---Chunk {i+1}----")
    print(recursive_chunk.page_content)
    print()

[209, 409, 463, 79, 478, 373, 378, 411, 433, 378]
3012
---Chunk 1----
LAWS OF KENYA
THE ELECTION OFFENCES ACT
CAP. 66
Revised and published by the National Council for Law Reporting
with the authority of the Attorney-General as gazetted by the Government Printer
www.kenyalaw.org

---Chunk 2----
Kenya
Election Offences Act
Cap. 66
Legislation as at 31 December 2022
 
 
 
 
 
 
 
 
 
 
 
 
By Kenya Law and Laws.Africa. Share widely and freely.
www.kenyalaw.org | info@kenyalaw.org
FRBR URI: /akn/ke/act/2016/37/eng@2022-12-31
There is no copyright on the legislative content of this document.
This PDF copy is licensed under a Creative Commons Attribution NonCommercial ShareAlike 4.0 License (CC BY-NC-

---Chunk 3----
SA 4.0). This license enables reusers to distribute, remix, adapt, and build upon the material in any medium or format
for noncommercial purposes only, and only so long as attribution is given to the creator. If you remix, adapt, or build
upon the material, you must license the

### c) Semantic Splitting

In [77]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_splitter=SemanticChunker(OpenAIEmbeddings())

semantic_chunks=semantic_splitter.split_documents(documents)
semantic_chunk_lengths=[]
for chunk in semantic_chunks:
    semantic_chunk_lengths.append(len(chunk.page_content))
print(semantic_chunk_lengths)
print(len(semantic_chunks))

[209, 418, 534, 476, 752, 1815, 543, 1387, 522, 1979, 2254, 364, 1932, 1338, 1122, 2040, 1780, 750, 1815, 932, 3264, 20, 1199, 1767, 2152, 958, 1386, 1941, 1683, 10, 143, 39, 78, 280, 232, 1065, 1189, 1286, 1102, 134, 2390, 233, 463, 2232, 1877, 897, 173, 1780, 488, 87, 588, 1546, 425, 350, 1155, 450, 1227, 294, 807, 227, 72, 317, 954, 547, 603, 31, 38, 79, 792, 813, 690, 991, 127, 372, 1567, 322, 1597, 570, 1293, 684, 1109, 51, 801, 220, 1625, 852, 613, 183, 1619, 2255, 525, 1358, 693, 1042, 334, 1829, 107, 413, 886, 865, 620, 1188, 292, 699, 657, 959, 253, 672, 113, 1493, 832, 1280, 351, 793, 700, 260, 845, 314, 106, 261, 2015, 2523, 2295, 161, 2601, 2434, 2621, 2669, 2627, 2304, 979, 1535, 2498, 2571, 2537, 1744, 842, 1062, 3190, 3030, 3197, 3258, 1218, 764, 461, 135, 371, 302, 123, 467, 230, 229, 355, 1687, 595, 1190, 513, 1291, 895, 314, 573, 398, 498, 538, 0, 565, 139, 425, 192, 997, 152, 657, 576, 91, 747, 230, 395, 1001, 40, 321, 552, 637, 719, 265, 833, 182, 900, 307, 1029, 96

- We observe that the chunk length is much more efficient in terms of memory consumption

In [78]:
for i, chunk in enumerate(semantic_chunks):
    print (f"------Chunk {i+1}-------")
    print(chunk.page_content)
    print()

------Chunk 1-------
LAWS OF KENYA
THE ELECTION OFFENCES ACT
CAP. 66
Revised and published by the National Council for Law Reporting
with the authority of the Attorney-General as gazetted by the Government Printer
www.kenyalaw.org

------Chunk 2-------
Kenya
Election Offences Act
Cap. 66
Legislation as at 31 December 2022
 
 
 
 
 
 
 
 
 
 
 
 
By Kenya Law and Laws.Africa. Share widely and freely. www.kenyalaw.org | info@kenyalaw.org
FRBR URI: /akn/ke/act/2016/37/eng@2022-12-31
There is no copyright on the legislative content of this document. This PDF copy is licensed under a Creative Commons Attribution NonCommercial ShareAlike 4.0 License (CC BY-NC-
SA 4.0).

------Chunk 3-------
This license enables reusers to distribute, remix, adapt, and build upon the material in any medium or format
for noncommercial purposes only, and only so long as attribution is given to the creator. If you remix, adapt, or build
upon the material, you must license the modiﬁed material under identical ter

### d) Structure Aware Splitting

In [36]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_text= """
# Library Handbook

## Hours of Operation
The library opens at 8:00 AM on weekdays and 10:00 AM on weekends. It closes at 9:00 PM every day.

## Borrowing
Students may borrow up to 8 books for 3 weeks. Faculty may borrow up to 20 books for a semester.

# Budget Committee Notes

## March Approval
The budget was approved in March after debate in committee. The vote was 14 to 3. The approved amount was $42,000, allocated to periodicals and digital subscriptions.

# Study Room Policy

## Reservations
Group study rooms can be reserved online for up to 2 hours.

## Restrictions
Food is not permitted; covered drinks are allowed.
"""

headers_to_split_on=[("#","Header 1"), ("##","Header 2")]
structure_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
structure_chunks = structure_splitter.split_text(markdown_text)

for i, chunk in enumerate(structure_chunks):
    print(f"----chunk {i+1}-----")
    print("Metadata:",chunk.metadata)
    print(chunk.page_content)
    print()

----chunk 1-----
Metadata: {'Header 1': 'Library Handbook', 'Header 2': 'Hours of Operation'}
The library opens at 8:00 AM on weekdays and 10:00 AM on weekends. It closes at 9:00 PM every day.

----chunk 2-----
Metadata: {'Header 1': 'Library Handbook', 'Header 2': 'Borrowing'}
Students may borrow up to 8 books for 3 weeks. Faculty may borrow up to 20 books for a semester.

----chunk 3-----
Metadata: {'Header 1': 'Budget Committee Notes', 'Header 2': 'March Approval'}
The budget was approved in March after debate in committee. The vote was 14 to 3. The approved amount was $42,000, allocated to periodicals and digital subscriptions.

----chunk 4-----
Metadata: {'Header 1': 'Study Room Policy', 'Header 2': 'Reservations'}
Group study rooms can be reserved online for up to 2 hours.

----chunk 5-----
Metadata: {'Header 1': 'Study Room Policy', 'Header 2': 'Restrictions'}
Food is not permitted; covered drinks are allowed.



## 3. Embedding and Storing

In [79]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [80]:
my_persist_directory="db/chroma_db"
vector_store=Chroma.from_documents(documents=semantic_chunks,
                                   embedding=embeddings,
                                   persist_directory=my_persist_directory,
                                  )
print("Vector Store Created")


Vector Store Created


In [81]:
question="What does the UDA manifesto say?"
matches=vector_store.similarity_search(question, k=3)

for match in matches:
    print(match.page_content[:180])


Two reasons. First, to earn a place at the table, one must first 
emerge as the undisputed kingpin of his com-
munity, by finishing all other contenders for the 
throne.Second it i
3
T
he election ahead of us is a pivotal one. We 
have three issues on the ballot and how 
we chose will determine whether Kenya 
marches forward or backwards. My first agenda is t
91. (1) Every political party shall—
(a)  have  a  national  character as  prescribed  by  an  Act  of 
Parliament;
(b) have a democratically elected governing body;
(c) promote an


In [54]:
from langchain_openai import ChatOpenAI

In [82]:
context="\n\n".join(match.page_content for match in matches)
model= ChatOpenAI(model="gpt-4o-mini")
prompt = f"""
Answer the question using only the context below.
If the context does not contain the answer, say so:

Context:
{context}

Question: {question}

"""
answer = model.invoke(prompt)
print(answer.content)

The UDA manifesto emphasizes the importance of political inclusion and economic empowerment for all Kenyans, advocating for national democratic political parties that allow citizens to participate fully in decision-making and hold leaders accountable. It rejects the need for patrons and political gatekeepers, promoting equal opportunities for leadership based on ability and desire to serve, regardless of tribe, gender, physical disability, or social background. Additionally, it highlights the need for jobs, stating that many young Kenyans struggle to find stable employment, and outlines a commitment to defending the Constitution, empowering the judiciary, and institutionalizing politics to reduce the personalization of political power.
